# Sandbox

**This notebook is not part of the project.** Nothing depends on it, `pipeline.py`
never runs it, and `check.py` never tests it. It exists so you have somewhere to
poke at the data without breaking anything.


### Before you run anything

Top right → **Select Kernel** → **Python Environments** → the one inside `.venv`.
If you pick the system Python 3.9 instead, the first import fails.

 → **Select Kernel** → **Python Environments** →  `.venv` 。

`Shift + Enter` runs a cell.

### One habit

Clear all outputs before you commit (`...` menu → Clear All Outputs). A notebook
full of saved output makes `git diff` unreadable, and this one is committed.



In [ ]:
# A notebook has no __file__, so we point at the project root explicitly.
# notebook 没有 __file__，所以直接指明项目根目录。
import os, sys
ROOT = os.path.expanduser("~/tim")
os.chdir(ROOT)                                     # so "data/..." resolves
sys.path[:0] = [ROOT, os.path.join(ROOT, "lib")]   # config.py in ROOT, modules in lib/

import numpy as np
import pandas as pd

import analysis, config, engine, timlib as tl
from strategies import STRATEGIES, BY_ID

px = tl.to_monthly(tl.load_prices())
vol = tl.to_monthly(tl.load_prices(column="Volume"), how="mean")
dates = list(px.index)
spy = engine.benchmark_returns(px, dates)

print(px.shape, dates[0].date(), "->", dates[-1].date())
px.tail(3).iloc[:, :6]

## Things worth trying

Each of these is one cell. Edit them, break them, see what happens.


In [ ]:
# 1. Which stocks did best, and how bumpy was the ride?
#    哪些股票表现最好？路有多颠？
rets = tl.returns(px)
table = pd.DataFrame({
    "total": px.iloc[-1] / px.iloc[0] - 1,
    "ann_vol": rets.std() * np.sqrt(12),
    "sharpe": rets.mean() / rets.std() * np.sqrt(12),
    "sector": pd.Series(config.SECTORS),
}).dropna().sort_values("sharpe", ascending=False)
table.head(12).style.format({"total": "{:.0%}", "ann_vol": "{:.1%}", "sharpe": "{:.2f}"})

In [ ]:
# 2. Write your OWN strategy and run it through the engine.
#    写一个你自己的策略，扔进引擎里跑。
#    A strategy is just a score: one number per stock, per month. Higher = we like it.
#    策略就是一个打分：每只股票每个月一个数，越高越喜欢。

my_score = lambda p, v, i: p.iloc[i] / p.iloc[i - 9] - 1        # 9-month momentum

run = engine.backtest(px, vol, dates, my_score, mode="long_only")
m = tl.metrics(run["net"])
ic = analysis.ic_summary(analysis.ic_series(px, vol, dates, my_score))

print(f"Sharpe {m['sharpe']:.2f}   CAGR {tl.pct(m['cagr'])}   maxDD {tl.pct(m['max_drawdown'])}")
print(f"beta {tl.beta_of(run['net'], spy):.2f}   turnover {run['avg_turnover']:.2f}")
print(f"rank IC {ic['mean']:+.4f}   t {ic['t']:+.2f}   <- if |t| < 2 it is noise")
print(f"\nSPY over the same window: Sharpe {tl.metrics(spy)['sharpe']:.2f}")

# Now try the long-short version. The Sharpe will drop a lot. That is the lesson.
# 现在试试多空版本。夏普会掉很多。那就是这一课的要点。
ls = tl.metrics(engine.backtest(px, vol, dates, my_score, mode="long_short")["net"])
print(f"long-short: Sharpe {ls['sharpe']:.2f}   CAGR {tl.pct(ls['cagr'])}")

In [ ]:
# 3. Does the signal work better in some sectors than others?
#    这个信号在某些行业里是不是更有效？
#    (One of the four follow-ups listed in RESULTS.md starts here.)
#    （RESULTS.md 里列的四个后续方向，其中一个就从这里开始。）
sector_ic = {}
for sector, names in config.UNIVERSE_BY_SECTOR.items():
    have = [t for t in names if t in px.columns]
    if len(have) < 8:
        continue
    # min_names=6 because a single sector has ~10 stocks, not the ~109 the default assumes.
    # min_names=6，因为单个行业只有约 10 只股票，不是默认值假设的约 109 只。
    series = analysis.ic_series(px[have], vol[have], dates,
                                BY_ID["MOM12_1"]["score"], min_names=6)
    s = analysis.ic_summary(series)
    sector_ic[sector] = {"stocks": len(have), "IC": round(s["mean"], 4), "t": round(s["t"], 2)}

pd.DataFrame(sector_ic).T.sort_values("IC", ascending=False)

In [ ]:
# 4. Query the database instead. Run steps/06_database_load.py first.
#    改用数据库查询。先跑 steps/06_database_load.py。
import duckdb

con = duckdb.connect(config.DUCKDB_FILE, read_only=True)
con.sql("""
    SELECT ticker,
           round(stddev_samp(ret) * sqrt(12) * 100, 1) AS ann_vol_pct,
           round(avg(ret) * 12 * 100, 1)               AS ann_ret_pct
    FROM monthly_returns
    WHERE ret IS NOT NULL
    GROUP BY ticker
    ORDER BY ann_vol_pct DESC
    LIMIT 10
""").df()

In [ ]:
# 5. Your turn. / 该你了。
